In [1]:
"""
The "Isekai" Guild Master - A Multi-Agent Quest Routing System
=================================================================

Four agents, wired into a cyclic state graph using raw Python (no
LangGraph/CrewAI dependency required):

    Guild Receptionist  -> Grand Library Agent -> [Guild Strategist <-> Guild Master] -> done

  1. Guild Receptionist  (Structured Output Agent)
     Extracts {"enemy_type", "environment", "urgency_level"} from a raw,
     panicked villager plea via prompt-engineered LLM call (falls back to a
     deterministic rule-based extractor if no ANTHROPIC_API_KEY is set).

  2. Grand Library Agent (Tool-Calling Agent)
     Calls search_monster_manual(enemy) -- a mock tool over a hardcoded
     Python dict of monster weaknesses -- and synthesizes the result.

  3. Guild Strategist    (Logic & Planning Agent)
     Selects a 3-person party from a hardcoded adventurer roster and drafts
     a 3-step battle plan exploiting the monster's weakness. On its first
     attempt it optimizes purely for offense/survivability, NOT safety --
     that's deliberately the Guild Master's job (see design note below).

  4. Guild Master        (Cyclic QA / Supervisor Agent)
     Reviews the plan against two safety rules:
       Rule 1: undead/shadow monsters require a Healer/Holy-caster.
       Rule 2: "High" urgency quests cannot be staffed by an all-Beginner party.
     If either rule fails, the Guild Master critiques the plan and loops the
     state back to the Strategist for at least one revision before the loop
     terminates (either via approval or a max_iterations cutoff).

Run this file directly to execute the 3 official assignment test cases.
"""

import os
import re
import json
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any


# ======================================================================
# SHARED STATE
# ======================================================================
@dataclass
class QuestState:
    """The shared state object passed between every agent in the graph."""
    original_query: str
    extraction: Optional[Dict[str, Any]] = None        # Guild Receptionist output
    tool_response: Optional[Dict[str, Any]] = None      # Grand Library Agent output
    party: Optional[List[Dict[str, Any]]] = None        # Guild Strategist output
    battle_plan: Optional[List[str]] = None             # Guild Strategist output
    guild_master_feedback: Optional[str] = None          # set when a plan is rejected
    force_include_healer: bool = False                   # constraint flag set by Guild Master
    force_avoid_all_beginners: bool = False               # constraint flag set by Guild Master
    approved: bool = False
    iteration_count: int = 0
    history: List[str] = field(default_factory=list)    # trace log of everything that happened

    def log(self, msg: str):
        self.history.append(msg)
        print(msg)


# ======================================================================
# MOCK DATABASE: THE GRAND LIBRARY'S MONSTER MANUAL
# ======================================================================
MONSTER_MANUAL = {
    "slime": {
        "display_name": "Giant Slime", "type": "Normal",
        "weak_to": ["Fire"], "immune_to": ["Physical"],
    },
    "shadow lich": {
        "display_name": "Shadow Lich", "type": "Undead",
        "weak_to": ["Holy", "Light"], "immune_to": ["Physical", "Shadow", "Dark"],
    },
    "goblin": {
        "display_name": "Goblin", "type": "Normal",
        "weak_to": ["Physical", "Fire"], "immune_to": [],
    },
    "shadow dragon": {
        "display_name": "Shadow Dragon", "type": "Undead-Dragon",
        "weak_to": ["Holy", "Light"], "immune_to": ["Shadow", "Dark", "Physical"],
    },
}


def search_monster_manual(enemy_type: str) -> Dict[str, Any]:
    """
    TOOL used by the Grand Library Agent.
    Fuzzy-matches the raw enemy_type string (as extracted by the Receptionist)
    against the hardcoded monster manual dict, e.g. "gelatinous
    physical-resistant slimes" -> "slime".
    """
    normalized = enemy_type.lower()
    for key, data in MONSTER_MANUAL.items():
        if key in normalized or any(word in normalized for word in key.split()):
            return {"matched_key": key, **data}
    return {"matched_key": None, "display_name": enemy_type,
            "type": "Unknown", "weak_to": [], "immune_to": []}


# ======================================================================
# MOCK DATABASE: AVAILABLE ADVENTURERS
# ======================================================================
ADVENTURERS = [
    {"name": "Megumin", "class": "Explosion Mage", "tags": ["Fire", "Magic"], "rank": "Intermediate"},
    {"name": "Darkness", "class": "Heavy Tank", "tags": ["Physical", "Tank"], "rank": "Advanced"},
    {"name": "Kirito", "class": "Dual Swordsman", "tags": ["Physical", "Dual-Wield"], "rank": "Advanced"},
    {"name": "Aqua", "class": "Healer", "tags": ["Holy", "Light", "Healing"], "rank": "Intermediate"},
    {"name": "Timmy", "class": "Novice Swordsman", "tags": ["Physical"], "rank": "Beginner"},
    {"name": "Village Guard", "class": "Beginner Guard", "tags": ["Physical"], "rank": "Beginner"},
]
HEALER_NAMES = {a["name"] for a in ADVENTURERS if "Healing" in a["tags"] or a["class"] == "Healer"}


# ======================================================================
# AGENT 1: THE GUILD RECEPTIONIST (Structured Output Agent)
# ======================================================================
EXTRACTION_SYSTEM_PROMPT = """You are the Guild Receptionist at a fantasy adventurer's guild.
A villager will describe an emergency in a panicked, unstructured way.
Extract exactly this JSON object and NOTHING else -- no prose, no markdown fences:

{"enemy_type": "<short description of the monster/threat>", "environment": "<where it's happening>", "urgency_level": "Low" | "Medium" | "High"}

Guidance for urgency_level:
- "High": imminent catastrophe, deadlines ("by midnight"), legendary/ancient threats, city-wide danger.
- "Medium": ongoing damage/danger but not an immediate catastrophe.
- "Low": minor nuisance, no real danger to people.

Respond with ONLY the JSON object."""


def extract_with_llm(raw_text: str) -> dict:
    """Real LLM call. Requires `pip install anthropic` and ANTHROPIC_API_KEY set."""
    import anthropic
    client = anthropic.Anthropic()
    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=200,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": raw_text}],
    )
    raw_output = response.content[0].text.strip()
    raw_output = re.sub(r"^```(json)?|```$", "", raw_output.strip()).strip()
    return json.loads(raw_output)


# ---- Rule-based fallback (used automatically when no API key is set) ----
ENEMY_KEYWORDS = ["slime", "shadow lich", "lich", "goblin", "dragon", "shadow dragon"]
HIGH_URGENCY_SIGNS = ["by midnight", "legendary", "awakened", "will fall", "capital"]
LOW_URGENCY_SIGNS = ["just sitting", "eating apples", "blocking the bridge"]


def extract_with_rules(raw_text: str) -> dict:
    text_l = raw_text.lower()

    enemy_type = raw_text  # keep full phrase for fuzzy matching downstream
    if not any(kw in text_l for kw in ENEMY_KEYWORDS):
        enemy_type = raw_text  # still pass full text; library agent's tool will report "Unknown"

    if "field" in text_l or "wheat" in text_l or "farm" in text_l:
        environment = "Farmland"
    elif "crypt" in text_l:
        environment = "Crypts"
    elif "mountain" in text_l:
        environment = "Mountains"
    elif "bridge" in text_l:
        environment = "Bridge"
    else:
        environment = "Unknown"

    if any(sign in text_l for sign in HIGH_URGENCY_SIGNS):
        urgency = "High"
    elif any(sign in text_l for sign in LOW_URGENCY_SIGNS):
        urgency = "Low"
    else:
        urgency = "Medium"

    return {"enemy_type": enemy_type, "environment": environment, "urgency_level": urgency}


def guild_receptionist(state: QuestState) -> QuestState:
    """Agent entry point: reads state.original_query, writes state.extraction."""
    if os.environ.get("ANTHROPIC_API_KEY"):
        try:
            extraction = extract_with_llm(state.original_query)
        except Exception as e:
            state.log(f"[Receptionist] LLM extraction failed ({e}), falling back to rules.")
            extraction = extract_with_rules(state.original_query)
    else:
        extraction = extract_with_rules(state.original_query)

    state.extraction = extraction
    state.log(f"[Guild Receptionist] Extracted: {extraction}")
    return state


# ======================================================================
# AGENT 2: THE GRAND LIBRARY AGENT (Tool-Calling Agent)
# ======================================================================
def grand_library_agent(state: QuestState) -> QuestState:
    """Agent entry point: reads state.extraction, writes state.tool_response."""
    enemy_type = state.extraction["enemy_type"]
    monster_data = search_monster_manual(enemy_type)  # <-- tool call

    synthesis = {
        "monster_name": monster_data["display_name"],
        "monster_type": monster_data["type"],
        "weak_to": monster_data["weak_to"],
        "immune_to": monster_data["immune_to"],
        "environment": state.extraction["environment"],
        "urgency_level": state.extraction["urgency_level"],
    }
    state.tool_response = synthesis
    state.log(f"[Grand Library Agent] search_monster_manual('{enemy_type[:40]}...') -> "
               f"matched '{monster_data['matched_key']}'")
    state.log(f"[Grand Library Agent] Synthesis: {synthesis}")
    return state


# ======================================================================
# AGENT 3: THE GUILD STRATEGIST (Logic & Planning Agent)
# ======================================================================
def _score_adventurer(adventurer, weak_to):
    """Higher score = better elemental/tag match against the monster's weaknesses."""
    return len(set(a.lower() for a in adventurer["tags"]) & set(w.lower() for w in weak_to))


def _select_party(weak_to, force_include_healer=False, force_avoid_all_beginners=False):
    rank_order = {"Advanced": 3, "Intermediate": 2, "Beginner": 1}

    def sort_key(a):
        return (_score_adventurer(a, weak_to), rank_order.get(a["rank"], 0))

    non_healers = sorted([a for a in ADVENTURERS if a["class"] != "Healer"], key=sort_key, reverse=True)
    healers = [a for a in ADVENTURERS if a["class"] == "Healer"]

    if force_include_healer and healers:
        party = healers[:1] + non_healers[:2]
    else:
        # First-pass behavior: purely offense/survivability-optimized, no healer bias
        party = non_healers[:3]

    if force_avoid_all_beginners and all(m["rank"] == "Beginner" for m in party):
        upgrade_candidates = [a for a in non_healers + healers
                               if a["rank"] != "Beginner" and a not in party]
        if upgrade_candidates:
            party[-1] = sorted(upgrade_candidates, key=sort_key, reverse=True)[0]

    return party


def _draft_battle_plan(monster_name, weak_to, party):
    weakness_str = " and ".join(weak_to) if weak_to else "no known weakness"
    attacker = party[0]["name"]
    support = party[1]["name"] if len(party) > 1 else "the party"
    third = party[2]["name"] if len(party) > 2 else "the party"
    return [
        f"Step 1: {attacker} opens the engagement exploiting the {monster_name}'s "
        f"weakness to {weakness_str}.",
        f"Step 2: {support} holds the front line / provides support so {attacker} can "
        f"attack safely without being overwhelmed.",
        f"Step 3: {third} finishes the engagement, mopping up remaining threats and "
        f"securing the {monster_name.lower()}'s defeat.",
    ]


def guild_strategist(state: QuestState) -> QuestState:
    """Agent entry point: reads state.tool_response (+ feedback flags), writes party/battle_plan."""
    weak_to = state.tool_response["weak_to"]
    party = _select_party(weak_to, state.force_include_healer, state.force_avoid_all_beginners)
    battle_plan = _draft_battle_plan(state.tool_response["monster_name"], weak_to, party)

    state.party = party
    state.battle_plan = battle_plan

    party_str = ", ".join(f"{p['name']} ({p['class']}, {p['rank']})" for p in party)
    state.log(f"[Guild Strategist] Attempt #{state.iteration_count + 1} party: {party_str}")
    for step in battle_plan:
        state.log(f"    {step}")
    return state


# ======================================================================
# AGENT 4: THE GUILD MASTER (Cyclic QA / Supervisor Agent)
# ======================================================================
def _rule_1_undead_needs_healer(state: QuestState):
    monster_type = state.tool_response["monster_type"].lower()
    if not ("undead" in monster_type or "shadow" in monster_type):
        return True, None
    has_healer_or_holy = any(p["name"] in HEALER_NAMES or "Holy" in p["tags"] for p in state.party)
    if has_healer_or_holy:
        return True, None
    return False, (
        f"Rule 1 VIOLATION: The party is facing an undead/shadow monster "
        f"({state.tool_response['monster_name']}) but includes no Healer or "
        f"Holy-magic user. Undead encounters require holy support -- revise the party."
    )


def _rule_2_high_urgency_no_all_beginners(state: QuestState):
    if state.tool_response["urgency_level"] != "High":
        return True, None
    if all(p["rank"] == "Beginner" for p in state.party):
        return False, (
            "Rule 2 VIOLATION: Urgency is High, but the party consists entirely of "
            "Beginner-rank adventurers. A high-urgency mission cannot be staffed "
            "only with beginners -- revise the party."
        )
    return True, None


def guild_master(state: QuestState) -> QuestState:
    """Agent entry point: reviews state.party/battle_plan, sets approved/feedback flags."""
    state.iteration_count += 1
    ok1, critique1 = _rule_1_undead_needs_healer(state)
    ok2, critique2 = _rule_2_high_urgency_no_all_beginners(state)

    if ok1 and ok2:
        state.approved = True
        state.guild_master_feedback = None
        state.log(f"[Guild Master] Plan APPROVED on attempt #{state.iteration_count}. "
                   f"All safety rules satisfied.")
    else:
        state.approved = False
        state.guild_master_feedback = " ".join(c for c in (critique1, critique2) if c)
        if not ok1:
            state.force_include_healer = True
        if not ok2:
            state.force_avoid_all_beginners = True
        state.log(f"[Guild Master] Plan REJECTED on attempt #{state.iteration_count}: "
                   f"{state.guild_master_feedback}")
        state.log("[Guild Master] Looping back to the Guild Strategist for revision...")
    return state


# ======================================================================
# ORCHESTRATOR: THE CYCLIC STATE GRAPH
# ======================================================================
def run_mas(query: str, max_iterations: int = 3) -> QuestState:
    """
    Graph: Receptionist -> Library -> [Strategist <-> Guild Master] -> done

    The Strategist/Guild Master pair forms the required cyclic edge: a
    rejected plan loops back to the Strategist (with constraint flags set)
    for at least one revision before the loop can terminate.
    """
    state = QuestState(original_query=query)
    state.log(f"\n{'='*90}\nNEW QUEST: \"{query}\"\n{'='*90}")

    guild_receptionist(state)
    grand_library_agent(state)

    while True:
        guild_strategist(state)
        guild_master(state)
        if state.approved:
            break
        if state.iteration_count >= max_iterations:
            state.log(f"[Orchestrator] Max iterations ({max_iterations}) reached without "
                       f"approval. Escalating to a senior guild member.")
            break

    status = "APPROVED" if state.approved else "ESCALATED (unresolved)"
    state.log(f"\n>>> FINAL STATUS: {status} after {state.iteration_count} attempt(s).")
    if state.approved:
        final_party = ", ".join(f"{p['name']} ({p['class']})" for p in state.party)
        state.log(f">>> Final party: {final_party}")

    return state


# ======================================================================
# MAIN: RUN THE 3 OFFICIAL ASSIGNMENT TEST CASES
# ======================================================================
if __name__ == "__main__":
    TEST_CASES = [
        "Help! A swarm of gelatinous physical-resistant slimes is destroying our wheat fields! They are everywhere!",
        "The legendary Shadow Lich has awakened in the crypts! The capital will fall by midnight!",
        "A giant goblin is blocking the bridge. It's just sitting there eating apples.",
    ]
    for query in TEST_CASES:
        run_mas(query)


NEW QUEST: "Help! A swarm of gelatinous physical-resistant slimes is destroying our wheat fields! They are everywhere!"
[Guild Receptionist] Extracted: {'enemy_type': 'Help! A swarm of gelatinous physical-resistant slimes is destroying our wheat fields! They are everywhere!', 'environment': 'Farmland', 'urgency_level': 'Medium'}
[Grand Library Agent] search_monster_manual('Help! A swarm of gelatinous physical-res...') -> matched 'slime'
[Grand Library Agent] Synthesis: {'monster_name': 'Giant Slime', 'monster_type': 'Normal', 'weak_to': ['Fire'], 'immune_to': ['Physical'], 'environment': 'Farmland', 'urgency_level': 'Medium'}
[Guild Strategist] Attempt #1 party: Megumin (Explosion Mage, Intermediate), Darkness (Heavy Tank, Advanced), Kirito (Dual Swordsman, Advanced)
    Step 1: Megumin opens the engagement exploiting the Giant Slime's weakness to Fire.
    Step 2: Darkness holds the front line / provides support so Megumin can attack safely without being overwhelmed.
    Step 3: Kiri